# GMO Case Study: Estimating the Equity Risk Premium

**Case:** Grantham, Mayo, and Van Otterloo, 2012: *Estimating the Equity Risk Premium* (HBS 9-213-051)
**Data:** `gmo_returns_weekly.xlsx` (weekly returns)

This notebook answers Sections 2–6 of the GMO case assignment step by step.
We try to keep things simple: every calculation has a short note on **what** we do and **why**.

**Contents** (the reading questions in Section 1 are not included)
2. Analyzing GMO's flagship fund (GMWAX)
3. Other GMO funds
4. GMO across subsamples
5. Factor exposures
6. Forecasts

---
# 2. Analyzing GMO's Flagship Fund (GMWAX)

## Setup: load the data

The Excel file has these sheets:
- `gmo`: weekly total returns on the 4 GMO funds
- `benchmarks`: ETFs for many asset classes (SPY, EFA, EEM, TLT, ...)
- `indexes`: MSCI ACWI (`MXWD`) and the Bloomberg US Aggregate bond index (`LBUSTRUU`)
- `rates`: the 3-month T-bill yield (`USGG3M`), which we use as the risk-free rate
- `info`: descriptions of each ticker

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.options.display.float_format = "{:.4f}".format

filename = "gmo_returns_weekly.xlsx"

gmo        = pd.read_excel(filename, sheet_name="gmo",        index_col=0, parse_dates=True)
benchmarks = pd.read_excel(filename, sheet_name="benchmarks", index_col=0, parse_dates=True)
indexes    = pd.read_excel(filename, sheet_name="indexes",    index_col=0, parse_dates=True)
rates      = pd.read_excel(filename, sheet_name="rates",      index_col=0, parse_dates=True)
info       = pd.read_excel(filename, sheet_name="info").set_index("Ticker")

ANN = 52   # weekly data -> 52 weeks per year

info[["Name"]]

### Build returns, the GBI, and excess returns

- **GBI (GMWAX Benchmark Index)** = 65% MSCI ACWI (`MXWD`) + 35% US Aggregate bonds (`LBUSTRUU`).
- **Risk-free rate:** `USGG3M` is an *annual* yield (0.05 means 5% per year). To get a weekly rate, we divide by 52.
- **Excess return** = fund return - risk-free return. Sharpe ratios and regressions use excess returns.

In [ ]:
# weekly risk-free rate
rf = (rates["USGG3M"] / ANN).rename("RF")

# put all total returns in one table (dates follow the GMO funds)
rets = gmo.join(benchmarks, how="left").join(indexes, how="left")

# the GMWAX benchmark index
rets["GBI"] = 0.65 * rets["MXWD"] + 0.35 * rets["LBUSTRUU"]

# excess returns = total return - risk-free rate
rf = rf.reindex(rets.index)
xrets = rets.sub(rf, axis=0)

# when does each fund start?
rets[["GMWAX", "GBMFX", "GQETX", "GMGEX"]].apply(lambda col: col.first_valid_index())

### Helper functions

We will run the same calculations many times, so we write them once as functions.

- `summary_stats`: annualized mean, volatility, Sharpe ratio, plus minimum return, 5th percentile (5% VaR), and maximum drawdown.
   - Annualized mean = weekly mean × 52
   - Annualized vol = weekly std × √52
   - Sharpe = annualized mean / annualized vol
   - **Max drawdown** = the worst fall from a previous peak in cumulative value
- `ols`: a linear regression $y_t = \alpha + \beta' x_t + \epsilon_t$ (the "LFD", linear factor decomposition). It reports the annualized alpha, the betas, the R², and the **information ratio** (alpha / residual vol, both annualized).

In [ ]:
def max_drawdown(r):
    wealth = (1 + r.dropna()).cumprod()     # growth of $1
    peak = wealth.cummax()                  # highest value so far
    drawdown = wealth / peak - 1            # % below the peak
    return drawdown.min()


def summary_stats(df):
    out = pd.DataFrame(index=df.columns)
    out["Mean"]   = df.mean() * ANN
    out["Vol"]    = df.std() * np.sqrt(ANN)
    out["Sharpe"] = out["Mean"] / out["Vol"]
    out["Min"]    = df.min()
    out["5% VaR"] = df.quantile(0.05)
    out["Max DD"] = df.apply(max_drawdown)
    return out


def ols(y, X):
    # regress y on the columns of X (with an intercept)
    data = pd.concat([y, X], axis=1).dropna()           # keep dates where all series exist
    yv = data.iloc[:, 0].values
    Xv = np.column_stack([np.ones(len(data)), data.iloc[:, 1:].values])
    coef, *_ = np.linalg.lstsq(Xv, yv, rcond=None)      # least squares
    resid = yv - Xv @ coef

    out = {"Alpha (ann.)": coef[0] * ANN}
    for name, b in zip(data.columns[1:], coef[1:]):
        out[f"Beta {name}"] = b
    out["R-squared"]  = 1 - resid.var() / yv.var()
    out["Info Ratio"] = coef[0] * ANN / (resid.std(ddof=1) * np.sqrt(ANN))
    out["Obs"] = len(data)
    return pd.Series(out)

## 2.1 Benchmarks

Summary statistics for the excess returns of GMWAX, GBI, and SPY over the full sample.

In [ ]:
cols = ["GMWAX", "GBI", "SPY"]
x2 = xrets[cols].dropna()
print("Sample:", x2.index[0].date(), "to", x2.index[-1].date())

summary_stats(x2)

In [ ]:
# correlation matrix
x2.corr()

## 2.2 Attribution

We run two univariate regressions:

$$ \tilde r^{GMWAX}_t = \alpha + \beta\, \tilde r^{SPY}_t + \epsilon_t \qquad\text{and}\qquad \tilde r^{GMWAX}_t = \alpha + \beta\, \tilde r^{GBI}_t + \epsilon_t $$

In [ ]:
attribution = pd.DataFrame({
    "GMWAX on SPY": ols(x2["GMWAX"], x2[["SPY"]]),
    "GMWAX on GBI": ols(x2["GMWAX"], x2[["GBI"]]),
}).T
attribution[["Alpha (ann.)", "Beta SPY", "Beta GBI", "R-squared", "Info Ratio"]]

## 2.3 Interpretation

**Relative performance overall:** GMWAX has a Sharpe ratio of about **0.47**, which is about the same as the GBI (0.48) and SPY (0.50). So on a risk-adjusted basis it is **not better** than simply holding the benchmark or the S&P 500. Its main advantage is **lower risk**: volatility of about 10% (vs. 18% for SPY) and a max drawdown of about **-32%** (vs. -41% for the GBI and -60% for SPY).

**Is it low-beta to the market?** **Yes.** The beta to SPY is only about **0.46**. A 1% move in SPY goes with only about a 0.46% move in GMWAX. This makes sense because it holds a lot of bonds.

**Does it track the index?** **Only loosely.** The correlation with the GBI is 0.86 and the R² is about **0.74**, so about a quarter of GMWAX's variation is *not* explained by its benchmark. The GBI beta is about 0.81 (below 1), so GMWAX takes less risk than its benchmark. It is an actively managed fund, not an index tracker.

**Does it earn alpha?** **Very little.** The alpha is about **0.8% per year** vs. SPY and **0.5% per year** vs. the GBI, and the information ratios are only around 0.1. That is small and not convincing, especially after a manager fee of about 0.5%.

---
# 3. Other GMO Funds

## 3.1 Correlations and cumulative returns

In [ ]:
funds = ["GMWAX", "GBMFX", "GQETX", "GMGEX"]
cols3 = funds + ["SPY", "MXWD", "LBUSTRUU"]

xrets[cols3].corr()

In [ ]:
# growth of $1 (total returns), starting from the date when ALL series are available
common = rets[cols3].dropna()
cum = (1 + common).cumprod()

cum.plot(figsize=(11, 5), title=f"Cumulative total return, growth of $1 (from {common.index[0].date()})")
plt.ylabel("Value of $1")
plt.grid(alpha=0.3)
plt.show()

## 3.2 Repeat Section 2 for all GMO funds

Summary statistics, then a regression of each fund on SPY. We also regress **GMGEX on MXWD** because MSCI ACWI is its listed benchmark. *(Each fund uses its own available history.)*

In [ ]:
summary_stats(xrets[funds + ["SPY"]])

In [ ]:
reg_spy = pd.DataFrame({f: ols(xrets[f], xrets[["SPY"]]) for f in funds}).T
reg_spy

In [ ]:
# GMGEX vs its listed benchmark (MSCI ACWI)
ols(xrets["GMGEX"], xrets[["MXWD"]]).to_frame("GMGEX on MXWD").T

## 3.3 Comparing the GMO funds

**How similar are they?** **Very similar.** The funds are highly correlated with each other (about 0.76–0.94) and with world equities (MXWD, about 0.82–0.90), and almost uncorrelated with the US bond index. They are all mostly driven by global stocks. They differ mainly in **how much** equity risk they take:
- **GBMFX** (Benchmark-Free): lowest risk, with vol of about 8% and a SPY beta of about 0.34
- **GMWAX** (Global Asset Allocation): SPY beta of about 0.46
- **GQETX** (Quality) and **GMGEX** (Global Equity): almost pure equity funds, with SPY betas of about 0.75–0.80

**Which looks most attractive?**
- **GBMFX** has the highest Sharpe ratio (about **0.71**), the lowest volatility, the smallest drawdown (about -24%), and the largest alpha vs. SPY (about **2.1% per year**, IR of about 0.40). For a risk-averse investor it is the most attractive.
- **GQETX** has the highest average return (about 9.5%), a Sharpe ratio of about 0.64 (better than SPY), and positive alpha (about 1.5%). It is attractive for an investor who wants equity exposure.
- **GMGEX** is the weakest: Sharpe of about 0.19 and **negative alpha** (about -3.5% per year vs. SPY, -3% vs. its benchmark MXWD).

Our pick is **GBMFX**, with GQETX as the equity alternative. But Section 4 shows that GBMFX's strong record comes mostly from before 2012.

---
# 4. GMO Across Subsamples

We split the data into two periods: **start to 2011** (the period covered by the case) and **2012 to end** (after the case).

## 4.1 Sharpe ratios by subsample

In [ ]:
early = xrets.loc[:"2011"]
late  = xrets.loc["2012":]

sharpe_sub = pd.DataFrame({
    "Sharpe: start-2011": summary_stats(early[funds + ["SPY"]])["Sharpe"],
    "Sharpe: 2012-end":   summary_stats(late[funds + ["SPY"]])["Sharpe"],
})
sharpe_sub

**Start to 2011:** GMWAX (0.44) and especially GBMFX (1.14) have **much higher** Sharpe ratios than SPY (0.23). This period includes two equity crashes (2000–2002 and 2008), when GMO's defensive positioning paid off. GMGEX does worse than SPY. GQETX is roughly equal to SPY, but it only starts in 2004.

**2012 to end:** SPY's Sharpe jumps to about **0.87**. Only GQETX (0.91) matches it. GMWAX (0.51), GBMFX (0.46), and GMGEX (0.41) are all clearly **below** SPY.

So the outperformance shows up **mostly in the first subsample**, not in both.

## 4.2 Information ratios vs. SPY by subsample

For each fund and period, we regress the fund's excess return on SPY's and compute IR = alpha / residual vol.

In [ ]:
ir_sub = pd.DataFrame({
    "Alpha: start-2011": {f: ols(early[f], early[["SPY"]])["Alpha (ann.)"] for f in funds},
    "IR: start-2011":    {f: ols(early[f], early[["SPY"]])["Info Ratio"]   for f in funds},
    "Alpha: 2012-end":   {f: ols(late[f],  late[["SPY"]])["Alpha (ann.)"]  for f in funds},
    "IR: 2012-end":      {f: ols(late[f],  late[["SPY"]])["Info Ratio"]    for f in funds},
})
ir_sub

**Start to 2011:** positive alpha and IR for GMWAX (IR about 0.39) and **GBMFX (IR about 1.5, which is very high)**. GQETX is about zero and GMGEX is negative.

**2012 to end:** the IRs turn **negative** for GMWAX, GBMFX, and GMGEX. Only GQETX keeps a small positive IR (about 0.29).

Same conclusion as 4.1: GMO's asset allocation funds added value **before 2012 but not after**. The only fund with (modest) positive alpha in both periods is the Quality fund, GQETX.

## 4.3 Cumulative returns, 2001–2010

In [ ]:
p = pd.concat([rets[funds + ["SPY", "TLT"]], rf], axis=1).loc["2001":"2010"]

cum_0110 = pd.DataFrame({
    "First date": p.apply(lambda c: c.first_valid_index().date()),
    "Cumulative return 2001-2010": (1 + p).prod() - 1,
})
cum_0110

In [ ]:
# growth of $1 over 2001-2010 (series that start later begin at $1 on their first date)
(1 + p.fillna(0)).cumprod().plot(figsize=(11, 5), title="Growth of $1, 2001-2010")
plt.grid(alpha=0.3)
plt.show()

**Is the case correct that equities underperformed in this period?** **Yes.**
- Over 2001–2010, **SPY returned only about 15% in total**, while just holding **T-bills (the risk-free rate) returned about 24%**. Stocks did *worse than cash* for a whole decade, the "lost decade".
- Long Treasuries (TLT) did much better (about +67%, even though our TLT data only starts in Aug 2002).
- GMO's allocation funds did very well: **GMWAX about +105%** and GBMFX about +131% (from 2003). This is the period that built GMO's reputation.
- *Note:* GBMFX, GQETX, and TLT start later than 2001, so their numbers cover a shorter period.

---
# 5. Factor Exposures

## 5.1 Multivariate regression of GMWAX

$$ \tilde r^{GMWAX}_t = \alpha + \beta_{SPY}\tilde r^{SPY}_t + \beta_{EFA}\tilde r^{EFA}_t + \beta_{EEM}\tilde r^{EEM}_t + \beta_{TLT}\tilde r^{TLT}_t + \beta_{TIP}\tilde r^{TIP}_t + \epsilon_t $$

- SPY = US stocks, EFA = developed international stocks, EEM = emerging market stocks, TLT = long US Treasuries, TIP = inflation-protected Treasuries.
- The sample starts in Dec 2003, when TIP data begins.

In [ ]:
factors = ["SPY", "EFA", "EEM", "TLT", "TIP"]

reg_multi = pd.DataFrame({
    "GMWAX": ols(xrets["GMWAX"], xrets[factors]),
}).T
reg_multi

## 5.2 How well do these factors explain GMWAX?

- **Very well:** the R² is about **0.89**, up from 0.61 with SPY alone. Most of GMWAX's movement is just a mix of these ETFs.
- The biggest exposures are **international stocks (EFA, about 0.28)** and **US stocks (SPY, about 0.16)**, plus some EEM and TIPS.
- **Outperformance?** **Not much.** The alpha is only about **0.65% per year**, with an IR of about 0.2. After controlling for its asset-class exposures, GMWAX adds little value. You could get similar returns by holding a fixed mix of cheap ETFs.

## 5.3 Multivariate regression of GBMFX

In [ ]:
reg_multi_gbmfx = pd.DataFrame({
    "GBMFX (full sample)": ols(xrets["GBMFX"], xrets[factors]),
    "GBMFX (start-2011)":  ols(xrets.loc[:"2011", "GBMFX"], xrets.loc[:"2011", factors]),
    "GBMFX (2012-end)":    ols(xrets.loc["2012":, "GBMFX"], xrets.loc["2012":, factors]),
}).T
reg_multi_gbmfx

**GBMFX:**
- The R² is lower (about **0.71**), so this fund is **less** explained by simple ETF exposures. That matches its "benchmark-free" style.
- Its exposures are small: mostly EFA (about 0.20), with some EEM and TIP, and **almost no SPY (about 0.04)**. GMO avoids US stocks here, consistent with its view that US equities are expensive.
- The full-sample alpha is about **2.3% per year** (IR of about 0.55), which looks good. But splitting the sample shows the alpha comes mostly from **before 2012**. This matches Section 4.

## 5.4 Rolling betas (2-year window)

We use **GMWAX**. A 2-year window is 104 weeks. For each week, we run the regression on the previous 104 weeks only and save the betas. This shows how the fund's exposures **change over time**.

In [ ]:
WINDOW = 104   # 2 years of weekly data
fund = "GMWAX"

data = xrets[[fund] + factors].dropna()
rolling_results = {}
for i in range(WINDOW, len(data) + 1):
    window_data = data.iloc[i - WINDOW:i]
    rolling_results[window_data.index[-1]] = ols(window_data[fund], window_data[factors])

rolling = pd.DataFrame(rolling_results).T
beta_cols = [f"Beta {f}" for f in factors]

rolling[beta_cols + ["R-squared"]].tail()

In [ ]:
# bar chart of the betas at the final date
final_betas = rolling[beta_cols].iloc[-1]
final_betas.plot(kind="bar", figsize=(7, 4), title=f"{fund} betas, 2-year window ending {rolling.index[-1].date()}")
plt.axhline(0, color="black", linewidth=0.8)
plt.ylabel("Beta")
plt.xticks(rotation=0)
plt.show()

final_betas

## 5.5 Exposures over time

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
rolling[beta_cols].plot(ax=axes[0], title=f"{fund}: rolling 2-year betas")
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].grid(alpha=0.3)
rolling["R-squared"].plot(ax=axes[1], title="Rolling R-squared", color="gray")
axes[1].grid(alpha=0.3)
plt.show()

In [ ]:
# average beta in each calendar year, to make the changes easier to read
rolling[beta_cols].groupby(rolling.index.year).mean()

**How has the strategy changed over time?** *(the numbers below are rough, read off the yearly table above)*
- **US stocks (SPY):** the exposure stays small, about 0.1–0.25. GMO has *never* held a lot of US equity in this fund, and it cut SPY exposure further around 2022–2023.
- **International stocks (EFA):** this is the largest exposure, and it **grew** from about 0.2 (2005–2012) to about **0.4** (2022–2025). GMO has moved toward non-US developed markets, which its models see as cheaper.
- **Emerging markets (EEM):** rose from about 0.05 (2009–2013) to about 0.15 (2015–2021), then fell back to about 0.05–0.1.
- **Bonds (TLT, TIP):** TLT is close to 0 most of the time. The TIP beta moves around a lot, which is also partly noise because TIP and TLT are highly correlated.
- **R²** is above 0.9 in most windows, so over any 2-year period the fund behaves like a mix of these ETFs, but the mix keeps **changing**. That is the *tactical asset allocation*: GMO shifts weights toward the asset classes its valuation models say are cheap.

---
# 6. Forecasts

## 6.1 GMO's July 2015 7-year forecasts vs. what actually happened

GMO forecast these 7-year annualized **real** returns in July 2015. We compare them with the **realized annualized nominal excess returns** from July 2015 to June 2022 (7 years).

We show two versions of the realized return:
- **Arithmetic mean:** weekly mean × 52
- **Compounded (geometric):** the annual growth rate you actually got over the 7 years

In [ ]:
forecasts = pd.Series({
    "SPY": -2.0, "EFA": -0.5, "EEM": -3.2, "TLT": -0.9, "TIP": 0.0,
    "IJR": -2.0, "GWX": -1.1, "EMB":  2.1, "BWX": -2.8,
}) / 100

# 7 years after July 2015
sub = xrets.loc["2015-07-01":"2022-06-30", forecasts.index]
print("Period:", sub.index[0].date(), "to", sub.index[-1].date(), "|", len(sub), "weeks")

compare = pd.DataFrame({
    "GMO forecast (real)":          forecasts,
    "Realized mean (excess)":       sub.mean() * ANN,
    "Realized compounded (excess)": (1 + sub).prod() ** (ANN / len(sub)) - 1,
})
compare["Error (realized mean - forecast)"] = compare["Realized mean (excess)"] - compare["GMO forecast (real)"]
# show the table in percent
(compare * 100).round(1)

In [ ]:
compare[["GMO forecast (real)", "Realized mean (excess)"]].plot(kind="bar", figsize=(10, 4),
                                                                title="GMO July 2015 forecast vs realized (2015-2022)")
plt.axhline(0, color="black", linewidth=0.8)
plt.xticks(rotation=0)
plt.show()

print("Correlation between forecast and realized:",
      round(compare["GMO forecast (real)"].corr(compare["Realized mean (excess)"]), 2))

## 6.2 Discussion

**Are we surprised by GMO's forecast accuracy?**
- GMO's 2015 forecast was **poor**. GMO predicted negative real returns for almost everything, especially **US stocks (-2%)** and **US small caps (-2%)**. In fact, **SPY earned about +11% per year** in excess of cash and IJR about +10%. Most assets beat their forecasts by a lot. The correlation between forecast and realized returns is about **-0.3** (it got the ranking wrong too), compared with **+0.86** in the famous 2001 forecast (Exhibit 1).
- We are **not very surprised**. Return forecasts that look great in-sample usually do much worse out-of-sample. A 7-year forecast gives very few independent observations, so one great call (2001) can be partly luck. High valuations can also stay high (or go higher) for a long time.
- *(Small caveat: GMO forecast real total returns and we measure nominal excess returns. Inflation and T-bill rates were both low over most of 2015–2022, so this does not change the conclusion.)*

**Could a dynamic strategy based on these signals have avoided the 2000–2011 equity underperformance?**
- **Partly, yes.** In the late 1990s valuations were extremely high, and GMO's signals said US stocks would do badly. Funds that followed this signal (GMWAX, GBMFX) did very well over 2001–2010, as we saw in 4.3, while SPY lagged cash.
- **But** the signal was "early": GMO was underweight from 1997 and lost about 60% of its business before being proven right. A real-time strategy must survive that period. The same signal also said "stay out of US stocks" after 2012, and that cost a lot (Section 4 and 6.1). So the signal timed the 2000s well but not the 2010s.

**How does in-sample vs. out-of-sample relate to GMO's reliance on long-run mean reversion?**
- GMO's method assumes P/E ratios and profit margins **revert to a fixed long-run level within about 7 years**. In-sample (using historical data to fit the model), this looks very strong, because we already know where valuations ended up.
- Out-of-sample, two things can go wrong. (1) Mean reversion can take **much longer** than 7 years. (2) The "normal" level itself can **shift**: after 2012, US profit margins and P/E ratios stayed high for a long time (lower interest rates, more profitable tech companies). If the steady state moves, a model that assumes fixed mean reversion will keep predicting a crash that does not come.
- So long-horizon valuation signals have **real but noisy** predictive power. They look much better in-sample than they are in practice, which is why GMO itself says its forecasts are "noisy" and usually makes only modest bets.